# Phase 4: Model Training & Evaluation (Forensic Metrics)

## Overview
This notebook trains machine learning models to detect NTFS timestomping using ground truth labels from `Suspicious Files.csv`.

## Metric Philosophy

In forensic triage, **missing evidence is unacceptable**. Our metric framework reflects this:

### Primary Metrics (Model Selection)
- **Recall (Sensitivity)**: Hard constraint - must detect ALL known timestomped files
- **CRR (Candidate Reduction Rate)**: Percentage of files analyst does NOT need to inspect
- **NNI (Number Needed to Investigate)**: Files reviewed to find 1 real timestomp

### Secondary Metrics (Operational Assessment)
- **Recall@K**: Were all positives found in top K% of ranked results?
- **FPR (False Positive Rate)**: More honest than Precision under extreme imbalance

### Supplementary Metrics (Completeness)
- **F2 Score**: Recall weighted 4x more than Precision (asymmetric forensic costs)
- **AUCPR (Area Under Precision-Recall Curve)**: Imbalance-robust ranking metric

### Explicitly Excluded from Model Selection
- **F1 Score**: Reported for completeness only; penalizes recall equally with precision
- **ROC-AUC**: Hides false positive explosion; over-rewards trivial TN predictions

## Models
1. Random Forest
2. XGBoost
3. LightGBM
4. Logistic Regression

## Datasets
- Training: 22 datasets (12 PE + 10 APT/Malware)
- Validation: 5 datasets (02-APT19, 09-APT40, 12-Kimsuky, 13-Winnti731, LoneWolf)


In [1]:
# [Cell 2] Imports and Configuration

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Gradient Boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Imbalance handling
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Joblib for model saving
import joblib
import json

# =============================================================================
# DIRECTORY CONFIGURATION
# =============================================================================

PHASE3_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling/v1")
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version")
DATA_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ground truth file
GROUND_TRUTH_PATH = DATA_DIR / "Suspicious Files (v1).csv"

# Training datasets (22 total)
TRAINING_DATASETS = [
    "01-PE", "02-PE", "03-PE", "04-PE", "05-PE", "06-PE",
    "07-PE", "08-PE", "09-PE", "10-PE", "11-PE", "12-PE",
    "01-APT17", "02-APT19" ,"03-APT21", "04-APT28", "05-APT29", "06-APT30",
    "07-APT37", "08-APT38", "10-DarkHotel663", "11-DarkHotelbbd", "14-Winnti53b"
]

# Validation datasets (4 total)
VALIDATION_DATASETS = ["09-APT40","12-Kimsuky", "13-Winnti731", "LoneWolf"]

print("Libraries imported successfully.")
print(f"\nPhase 3 input: {PHASE3_DIR}")
print(f"Ground truth: {GROUND_TRUTH_PATH}")
print(f"Phase 4 output: {OUTPUT_DIR}")
print(f"\nTraining datasets: {len(TRAINING_DATASETS)}")
print(f"Validation datasets: {len(VALIDATION_DATASETS)}")

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, precision_recall_curve,
    average_precision_score, fbeta_score 
)


Libraries imported successfully.

Phase 3 input: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 3: Feature Engineering & Labeling/v1
Ground truth: /Users/soni/Github/Digital-Detectives_Thesis/data/Suspicious Files (v1).csv
Phase 4 output: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version

Training datasets: 23
Validation datasets: 4


## Step 1: Load Ground Truth Labels

Load the actual known timestomped files from `Suspicious Files.csv`.
This contains 44 files identified by Oh et al. using LogTracker tool.


In [2]:
# [Cell 4] Load Ground Truth

print("Loading ground truth labels...")

df_ground_truth = pd.read_csv(GROUND_TRUTH_PATH)
print(f"Ground truth loaded: {len(df_ground_truth)} known timestomped files")

print(f"\nGround truth structure:")
print(df_ground_truth.head(10))

print(f"\nTimestomped files per dataset:")
gt_per_dataset = df_ground_truth.groupby('dataID').size().reset_index(name='count')
print(gt_per_dataset.to_string(index=False))

# Check which datasets have ground truth
training_with_gt = [d for d in TRAINING_DATASETS if d in df_ground_truth['dataID'].values]
validation_with_gt = [d for d in VALIDATION_DATASETS if d in df_ground_truth['dataID'].values]

print(f"\nTraining datasets with ground truth: {len(training_with_gt)}/{len(TRAINING_DATASETS)}")
print(f"Validation datasets with ground truth: {len(validation_with_gt)}/{len(VALIDATION_DATASETS)}")


Loading ground truth labels...
Ground truth loaded: 64 known timestomped files

Ground truth structure:
  dataID                               FileName  is_timestomped
0  01-PE      NewFileTime_SI_C_Manipulation.dll             1.0
1  02-PE      NewFileTime_SI_M_Manipulation.dll             1.0
2  03-PE    NewFileTime_SI_MAC_Manipulation.dll             1.0
3  04-PE       PowerShell_SI_C_Manipulation.dll             1.0
4  05-PE       PowerShell_SI_M_Manipulation.dll             1.0
5  06-PE     PowerShell_SI_MAC_Manipulation.dll             1.0
6  07-PE       nTimestomp_SI_C_Manipulation.dll             1.0
7  08-PE       nTimestomp_SI_M_Manipulation.dll             1.0
8  09-PE    nTimestomp_SI_MACE_Manipulation.dll             1.0
9  11-PE  SetMACE_SI_MACE_Copy_Manipulation.dll             1.0

Timestomped files per dataset:
         dataID  count
       01-APT17      1
          01-PE      1
       02-APT19     13
          02-PE      1
       03-APT21      1
          03-PE      1

## Step 2: Load and Label Training Data

Load file features from Phase 3 and join with ground truth labels.
Handle the LoneWolf path format specially (ground truth uses `/Dropbox/filename.ext`).


In [3]:
# [Cell 6] Helper Function: Match Ground Truth

def match_ground_truth(df_features, df_gt, dataset_id):
    """
    Match ground truth labels to file features.
    
    For most datasets: match on FileName
    For LoneWolf: match on FilePath (ground truth uses /Dropbox/file.ext format)
    
    Returns DataFrame with is_timestomped column added.
    """
    df = df_features.copy()
    df['is_timestomped'] = 0  # Default: not timestomped
    
    # Get ground truth for this dataset
    gt_dataset = df_gt[df_gt['dataID'] == dataset_id]
    
    if len(gt_dataset) == 0:
        return df, 0  # No ground truth for this dataset
    
    matched_count = 0
    
    if dataset_id == "LoneWolf":
        # LoneWolf: ground truth uses path format like /Dropbox/DeathToll.jpg
        for _, gt_row in gt_dataset.iterrows():
            gt_path = gt_row['FileName']  # Contains path like /Dropbox/file.jpg
            
            if gt_path.startswith('/Dropbox/'):
                gt_filename = gt_path.split('/')[-1]
                mask = (df['FilePath'].str.contains('/Dropbox/', na=False)) & \
                       (df['FileName'] == gt_filename)
            else:
                mask = df['FilePath'] == gt_path
            
            matches = mask.sum()
            if matches > 0:
                df.loc[mask, 'is_timestomped'] = 1
                matched_count += matches
    else:
        # Other datasets: match on FileName directly
        for _, gt_row in gt_dataset.iterrows():
            gt_filename = gt_row['FileName']
            mask = df['FileName'] == gt_filename
            matches = mask.sum()
            if matches > 0:
                df.loc[mask, 'is_timestomped'] = 1
                matched_count += matches
    
    return df, matched_count


print("Ground truth matching function defined.")


Ground truth matching function defined.


In [4]:
# [Cell 7] Load and Label All Training Data

print("Loading and labeling training data...")
print("=" * 70)

all_training_data = []
training_label_summary = []

for dataset_id in TRAINING_DATASETS:
    features_path = PHASE3_DIR / f"file_features_{dataset_id}.csv"
    
    if not features_path.exists():
        print(f"  {dataset_id}: SKIPPED (file not found)")
        continue
    
    # Load features
    df_features = pd.read_csv(features_path, low_memory=False)
    
    # Match ground truth
    df_labeled, matched = match_ground_truth(df_features, df_ground_truth, dataset_id)
    
    # Get expected count from ground truth
    expected = len(df_ground_truth[df_ground_truth['dataID'] == dataset_id])
    
    print(f"  {dataset_id}: {len(df_labeled):,} files, {matched} timestomped (expected: {expected})")
    
    all_training_data.append(df_labeled)
    training_label_summary.append({
        'dataID': dataset_id,
        'total_files': len(df_labeled),
        'timestomped_found': matched,
        'timestomped_expected': expected
    })

# Combine all training data
df_train_all = pd.concat(all_training_data, ignore_index=True)

print("=" * 70)
print(f"\nTotal training files: {len(df_train_all):,}")
print(f"Total timestomped files found: {df_train_all['is_timestomped'].sum()}")

# Summary table
df_summary = pd.DataFrame(training_label_summary)
print(f"\nLabel matching summary:")
print(df_summary.to_string(index=False))


Loading and labeling training data...
  01-PE: 37,375 files, 1 timestomped (expected: 1)
  02-PE: 105,251 files, 1 timestomped (expected: 1)
  03-PE: 109,985 files, 1 timestomped (expected: 1)
  04-PE: 7,882 files, 1 timestomped (expected: 1)
  05-PE: 9,451 files, 1 timestomped (expected: 1)
  06-PE: 9,434 files, 1 timestomped (expected: 1)
  07-PE: 110,336 files, 1 timestomped (expected: 1)
  08-PE: 110,541 files, 1 timestomped (expected: 1)
  09-PE: 111,889 files, 1 timestomped (expected: 1)
  10-PE: 110,574 files, 0 timestomped (expected: 0)
  11-PE: 9,415 files, 1 timestomped (expected: 1)
  12-PE: 9,160 files, 1 timestomped (expected: 1)
  01-APT17: 31,455 files, 1 timestomped (expected: 1)
  02-APT19: 21,334 files, 23 timestomped (expected: 13)
  03-APT21: 28,028 files, 1 timestomped (expected: 1)
  04-APT28: 29,399 files, 1 timestomped (expected: 1)
  05-APT29: 29,860 files, 6 timestomped (expected: 6)
  06-APT30: 22,266 files, 1 timestomped (expected: 1)
  07-APT37: 29,426 file

In [5]:
# [Cell 8] Analyze Class Imbalance

print("=" * 70)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 70)

y_all = df_train_all['is_timestomped']

n_negative = (y_all == 0).sum()
n_positive = (y_all == 1).sum()

print(f"\nClass distribution:")
print(f"  Class 0 (Normal):     {n_negative:,} ({n_negative / len(y_all) * 100:.4f}%)")
print(f"  Class 1 (Timestomped): {n_positive:,} ({n_positive / len(y_all) * 100:.6f}%)")

if n_positive > 0:
    imbalance_ratio = n_negative / n_positive
    print(f"\nImbalance ratio: {imbalance_ratio:,.0f}:1")
    print(f"\nThis is EXTREME imbalance - traditional metrics like F1 and Precision")
    print(f"will appear poor even with excellent forensic performance.")
else:
    imbalance_ratio = float('inf')
    print("\nWARNING: No timestomped files found in training data!")


CLASS IMBALANCE ANALYSIS

Class distribution:
  Class 0 (Normal):     991,351 (99.9948%)
  Class 1 (Timestomped): 52 (0.005245%)

Imbalance ratio: 19,064:1

This is EXTREME imbalance - traditional metrics like F1 and Precision
will appear poor even with excellent forensic performance.


## Step 3: Prepare Features and Labels

Select feature columns and prepare the feature matrix.
Exclude identifier columns and any computed flag columns.


In [6]:
# [Cell 10] Select Features

# Define columns to exclude (identifiers and computed flags)
EXCLUDE_COLUMNS = [
    'FileFRN', 'dataID', 'FileName', 'FilePath',
    'is_timestomped',  # This is our target
    # Exclude any flag columns if they exist
    'flag_backward_timestamp', 'flag_creation_changed', 'flag_zero_nanoseconds',
    'flag_logfile_usn_mismatch', 'flag_usn_basic_pattern', 'flag_repeated_si_update',
    'flag_only_si_modified', 'flag_potential_timestomp', 'flag_silent_timestomp',
    'flag_high_suspicion', 'suspicion_score'
]

# Get feature columns
feature_columns = [col for col in df_train_all.columns if col not in EXCLUDE_COLUMNS]

# Create feature matrix
X = df_train_all[feature_columns].copy()

# Handle NaN values
X = X.fillna(0)

# Convert boolean columns to int
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)

# Target variable
y = df_train_all['is_timestomped'].astype(int)

print(f"Feature matrix shape: {X.shape}")
print(f"Number of features: {len(feature_columns)}")
print(f"\nFeature columns:")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i:2d}. {col}")


Feature matrix shape: (991403, 29)
Number of features: 29

Feature columns:
   1. num_timestamp_changes
   2. num_backward_jumps
   3. num_forward_jumps
   4. num_creation_changes
   5. max_backward_jump_seconds
   6. mean_jump_seconds
   7. timestamp_change_density
   8. num_zero_nanosecond_events
   9. only_SI_modified
  10. num_update_resident_value
  11. repeated_update_resident_value
  12. consecutive_timestamp_changes
  13. num_logfile_events
  14. has_logfile_ts_change
  15. has_usn_basic_info
  16. has_usn_close
  17. has_usn_file_create
  18. num_usn_basic_info
  19. num_usn_close
  20. num_usn_file_create
  21. logfile_usn_mismatch
  22. has_usn_basic_pattern
  23. num_usnjrnl_events
  24. min_inter_event_delta
  25. max_inter_event_delta
  26. mean_inter_event_delta
  27. burstiness_score
  28. event_time_span_seconds
  29. total_events


## Step 4: Train-Test Split

Split data into training (80%) and test (20%) sets using stratified sampling.
With extreme imbalance, stratification ensures both sets have representative samples.


In [7]:
# [Cell 12] Train-Test Split

if y.sum() < 2:
    print("ERROR: Not enough positive samples for train-test split!")
else:
    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    print("Train-Test Split Complete")
    print(f"\nTraining set: {len(X_train):,} samples")
    print(f"  Class 0 (Normal):     {(y_train == 0).sum():,} ({(y_train == 0).mean() * 100:.4f}%)")
    print(f"  Class 1 (Timestomped): {(y_train == 1).sum():,} ({(y_train == 1).mean() * 100:.6f}%)")
    
    print(f"\nTest set: {len(X_test):,} samples")
    print(f"  Class 0 (Normal):     {(y_test == 0).sum():,} ({(y_test == 0).mean() * 100:.4f}%)")
    print(f"  Class 1 (Timestomped): {(y_test == 1).sum():,} ({(y_test == 1).mean() * 100:.6f}%)")


Train-Test Split Complete

Training set: 793,122 samples
  Class 0 (Normal):     793,080 (99.9947%)
  Class 1 (Timestomped): 42 (0.005296%)

Test set: 198,281 samples
  Class 0 (Normal):     198,271 (99.9950%)
  Class 1 (Timestomped): 10 (0.005043%)


In [8]:
# [Cell 13] Handle Class Imbalance

print("=" * 70)
print("HANDLING EXTREME CLASS IMBALANCE")
print("=" * 70)

n_positive_train = (y_train == 1).sum()
n_negative_train = (y_train == 0).sum()

print(f"\nOriginal training distribution:")
print(f"  Positive: {n_positive_train}")
print(f"  Negative: {n_negative_train}")
print(f"  Ratio: {n_negative_train / n_positive_train:.0f}:1")

# SMOTE with controlled ratio
target_positive = min(n_negative_train // 100, n_positive_train * 100)
target_positive = max(target_positive, n_positive_train)

print(f"\nApplying SMOTE...")
print(f"  Target positive samples: {target_positive}")

if n_positive_train > 0:
    smote = SMOTE(
        sampling_strategy={1: target_positive},
        random_state=42,
        k_neighbors=min(5, n_positive_train - 1) if n_positive_train > 1 else 1
    )
    
    try:
        X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
        print(f"\nAfter SMOTE:")
        print(f"  Positive: {(y_train_balanced == 1).sum()}")
        print(f"  Negative: {(y_train_balanced == 0).sum()}")
        print(f"  New ratio: {(y_train_balanced == 0).sum() / (y_train_balanced == 1).sum():.0f}:1")
    except Exception as e:
        print(f"  SMOTE failed: {e}")
        X_train_balanced = X_train.copy()
        y_train_balanced = y_train.copy()
else:
    X_train_balanced = X_train.copy()
    y_train_balanced = y_train.copy()

# Class weights for algorithms
if n_positive_train > 0:
    weight_ratio = np.sqrt(n_negative_train / n_positive_train)
    class_weight_dict = {0: 1.0, 1: weight_ratio}
    scale_pos_weight = weight_ratio
else:
    class_weight_dict = {0: 1.0, 1: 1.0}
    scale_pos_weight = 1.0

print(f"\nClass weights: {class_weight_dict}")


HANDLING EXTREME CLASS IMBALANCE

Original training distribution:
  Positive: 42
  Negative: 793080
  Ratio: 18883:1

Applying SMOTE...
  Target positive samples: 4200

After SMOTE:
  Positive: 4200
  Negative: 793080
  New ratio: 189:1

Class weights: {0: 1.0, 1: np.float64(137.4149087357596)}


In [9]:
# [Cell 14] Feature Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

X_train_trees = X_train_balanced.values if hasattr(X_train_balanced, 'values') else X_train_balanced
X_test_trees = X_test.values if hasattr(X_test, 'values') else X_test

print("Feature scaling complete.")
print(f"Scaled training shape: {X_train_scaled.shape}")
print(f"Scaled test shape: {X_test_scaled.shape}")

# Save scaler
scaler_path = OUTPUT_DIR / "feature_scaler.joblib"
joblib.dump(scaler, scaler_path)
print(f"\nScaler saved to: {scaler_path}")


Feature scaling complete.
Scaled training shape: (797280, 29)
Scaled test shape: (198281, 29)

Scaler saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/feature_scaler.joblib


## Step 5: Define Forensic Metrics Framework

Define metrics appropriate for forensic triage where missing evidence is unacceptable.

### Primary Metrics (Model Selection)
- Recall: TP / (TP + FN) - Must be maximized
- CRR: 1 - (TP + FP) / Total - Workload reduction rate
- NNI: (TP + FP) / TP - Files to review per true positive

### Secondary Metrics (Operational)
- Recall@K: Were all positives in top K%?
- FPR: FP / (FP + TN) - False positive rate

### Supplementary Metrics (Completeness)
- F2 Score: Recall weighted 4x precision
- AUCPR: Area under precision-recall curve
- F1 Score: Reported only, not used for selection


In [10]:
# [Cell 16] Forensic Metrics Functions

def compute_forensic_metrics(y_true, y_pred, y_prob=None, total_files=None):
    """
    Compute comprehensive forensic metrics.
    
    Primary: Recall, CRR, NNI
    Secondary: FPR, Recall@K
    Supplementary: F2, AUCPR, F1 (for completeness)
    """
    if total_files is None:
        total_files = len(y_true)
    
    # Confusion matrix components
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
        if len(cm) == 1:
            if y_true.iloc[0] == 0 if hasattr(y_true, 'iloc') else y_true[0] == 0:
                tn = cm[0, 0]
            else:
                tp = cm[0, 0]
    
    # ===========================================
    # PRIMARY METRICS (Model Selection)
    # ===========================================
    
    # Recall (Sensitivity) - HARD CONSTRAINT
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    # CRR (Candidate Reduction Rate)
    # Percentage of files analyst does NOT need to inspect
    flagged_files = tp + fp
    crr = 1.0 - (flagged_files / total_files) if total_files > 0 else 0.0
    
    # NNI (Number Needed to Investigate)
    # Files reviewed to find 1 real timestomp
    nni = flagged_files / tp if tp > 0 else float('inf')
    
    # ===========================================
    # SECONDARY METRICS (Operational Assessment)
    # ===========================================
    
    # FPR (False Positive Rate) - more honest than Precision under imbalance
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    # Precision (for reference, but not primary)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    
    # ===========================================
    # SUPPLEMENTARY METRICS (Completeness)
    # ===========================================
    
    # F2 Score (Recall weighted 4x Precision)
    f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    
    # F1 Score (reported for completeness only)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # AUCPR (Area Under Precision-Recall Curve)
    if y_prob is not None and len(np.unique(y_true)) > 1:
        aucpr = average_precision_score(y_true, y_prob)
    else:
        aucpr = np.nan
    
    # Accuracy (for reference)
    accuracy = accuracy_score(y_true, y_pred)
    
    # Flag Rate
    flag_rate = flagged_files / total_files if total_files > 0 else 0.0
    
    metrics = {
        # Primary
        'Recall': recall,
        'CRR': crr,
        'NNI': nni,
        # Secondary
        'FPR': fpr,
        'Precision': precision,
        # Supplementary
        'F2': f2,
        'AUCPR': aucpr,
        'F1': f1,
        # Reference
        'Accuracy': accuracy,
        'Flag_Rate': flag_rate,
        # Confusion matrix
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn,
        'Total': total_files
    }
    
    return metrics


def compute_recall_at_k(y_true, y_prob, k_percentages=[1, 5, 10]):
    """
    Compute Recall@K - what percentage of positives are found in top K% of ranked results.
    """
    if y_prob is None:
        return {}
    
    n_total = len(y_true)
    n_positive = y_true.sum()
    
    if n_positive == 0:
        return {}
    
    # Sort by probability descending
    sorted_indices = np.argsort(y_prob)[::-1]
    y_true_sorted = np.array(y_true)[sorted_indices]
    
    recall_at_k = {}
    for k_pct in k_percentages:
        k = int(np.ceil(n_total * k_pct / 100))
        tp_at_k = y_true_sorted[:k].sum()
        recall_at_k[f'Recall@{k_pct}%'] = tp_at_k / n_positive
    
    return recall_at_k


def find_rank_of_all_positives(y_true, y_prob):
    """
    Find at what rank (percentage) all positives are captured.
    """
    if y_prob is None:
        return None
    
    n_total = len(y_true)
    n_positive = y_true.sum()
    
    if n_positive == 0:
        return None
    
    # Sort by probability descending
    sorted_indices = np.argsort(y_prob)[::-1]
    y_true_sorted = np.array(y_true)[sorted_indices]
    
    # Find index where cumulative positives equals total positives
    cumsum = np.cumsum(y_true_sorted)
    full_recall_idx = np.argmax(cumsum >= n_positive)
    
    # Convert to percentage
    rank_percentage = (full_recall_idx + 1) / n_total * 100
    
    return rank_percentage


print("Forensic metrics functions defined.")


Forensic metrics functions defined.


In [11]:
# [Cell 17] Evaluation Helper Functions

def evaluate_model(model, X_test, y_test, model_name, threshold=0.5, total_files=None):
    """
    Evaluate model with forensic metrics framework.
    """
    # Get probabilities
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= threshold).astype(int)
    else:
        y_pred = model.predict(X_test)
        y_prob = None
    
    # Compute forensic metrics
    metrics = compute_forensic_metrics(y_test, y_pred, y_prob, total_files)
    metrics['Model'] = model_name
    metrics['Threshold'] = threshold
    
    # Compute Recall@K
    recall_at_k = compute_recall_at_k(y_test, y_prob, [1, 5, 10, 20])
    metrics.update(recall_at_k)
    
    # Find rank for 100% recall
    rank_all = find_rank_of_all_positives(y_test, y_prob)
    metrics['Rank_100%_Recall'] = rank_all
    
    return metrics, y_pred, y_prob


def print_forensic_evaluation(metrics):
    """
    Print formatted forensic evaluation results.
    """
    print(f"\n{'='*70}")
    print(f"Model: {metrics['Model']} (threshold: {metrics['Threshold']:.2f})")
    print(f"{'='*70}")
    
    print(f"\n--- PRIMARY METRICS (Model Selection) ---")
    print(f"Recall:              {metrics['Recall']:.4f}  [HARD CONSTRAINT - must be 1.0]")
    print(f"CRR:                 {metrics['CRR']:.4f}  [{metrics['CRR']*100:.2f}% of files excluded from review]")
    print(f"NNI:                 {metrics['NNI']:.2f}   [review {metrics['NNI']:.1f} files per true positive]")
    
    print(f"\n--- SECONDARY METRICS (Operational) ---")
    print(f"FPR:                 {metrics['FPR']:.6f}  [false alarm rate among negatives]")
    print(f"Flag Rate:           {metrics['Flag_Rate']:.6f}  [{metrics['Flag_Rate']*100:.4f}% of files flagged]")
    
    if 'Rank_100%_Recall' in metrics and metrics['Rank_100%_Recall'] is not None:
        print(f"100% Recall at:      Top {metrics['Rank_100%_Recall']:.2f}% of ranked results")
    
    # Recall@K
    recall_k_keys = [k for k in metrics.keys() if k.startswith('Recall@')]
    if recall_k_keys:
        print(f"\n--- RECALL@K (Ranking Performance) ---")
        for k in sorted(recall_k_keys):
            print(f"{k}:           {metrics[k]:.4f}")
    
    print(f"\n--- SUPPLEMENTARY METRICS (Completeness) ---")
    print(f"F2 Score:            {metrics['F2']:.4f}  [recall-weighted harmonic mean]")
    if not np.isnan(metrics['AUCPR']):
        print(f"AUCPR:               {metrics['AUCPR']:.4f}  [area under PR curve]")
    print(f"F1 Score:            {metrics['F1']:.4f}  [reported only, NOT used for selection]")
    print(f"Accuracy:            {metrics['Accuracy']:.4f}  [misleading under imbalance]")
    
    print(f"\n--- CONFUSION MATRIX ---")
    print(f"TP (Detected):       {metrics['TP']:,}")
    print(f"FN (Missed):         {metrics['FN']:,}  [CRITICAL - must be 0]")
    print(f"FP (False alarms):   {metrics['FP']:,}")
    print(f"TN (Correct normal): {metrics['TN']:,}")


def find_recall_constrained_threshold(model, X_test, y_test, target_recall=1.0):
    """
    Find lowest threshold that achieves target recall.
    This maximizes CRR while maintaining recall constraint.
    """
    if not hasattr(model, 'predict_proba'):
        return 0.5
    
    y_prob = model.predict_proba(X_test)[:, 1]
    n_positive = y_test.sum()
    
    if n_positive == 0:
        return 0.5
    
    # Try thresholds from high to low
    best_threshold = 0.01
    for threshold in np.arange(0.99, 0.00, -0.01):
        y_pred = (y_prob >= threshold).astype(int)
        recall = recall_score(y_test, y_pred, zero_division=0)
        
        if recall >= target_recall:
            best_threshold = threshold
            break
    
    return best_threshold


print("Evaluation helper functions defined.")


Evaluation helper functions defined.


## Step 6: Model Training

Train 4 models optimized for HIGH RECALL (detecting all timestomped files).
Models are configured with class weights to prioritize minority class detection.


In [12]:
# [Cell 19] Train Random Forest

print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_model.fit(X_train_trees, y_train_balanced)

# Find recall-constrained threshold (maximize CRR at 100% recall)
rf_threshold = find_recall_constrained_threshold(rf_model, X_test_trees, y_test, target_recall=1.0)
print(f"Recall-constrained threshold: {rf_threshold:.2f}")

# Evaluate
rf_metrics, rf_pred, rf_prob = evaluate_model(
    rf_model, X_test_trees, y_test, "Random Forest", 
    threshold=rf_threshold, total_files=len(y_test)
)
print_forensic_evaluation(rf_metrics)

# Save model
rf_path = OUTPUT_DIR / "model_random_forest.joblib"
joblib.dump(rf_model, rf_path)
print(f"\nModel saved to: {rf_path}")


Training Random Forest...
Recall-constrained threshold: 0.08

Model: Random Forest (threshold: 0.08)

--- PRIMARY METRICS (Model Selection) ---
Recall:              1.0000  [HARD CONSTRAINT - must be 1.0]
CRR:                 0.9973  [99.73% of files excluded from review]
NNI:                 53.80   [review 53.8 files per true positive]

--- SECONDARY METRICS (Operational) ---
FPR:                 0.002663  [false alarm rate among negatives]
Flag Rate:           0.002713  [0.2713% of files flagged]
100% Recall at:      Top 0.27% of ranked results

--- RECALL@K (Ranking Performance) ---
Recall@1%:           1.0000
Recall@10%:           1.0000
Recall@20%:           1.0000
Recall@5%:           1.0000

--- SUPPLEMENTARY METRICS (Completeness) ---
F2 Score:            0.0865  [recall-weighted harmonic mean]
AUCPR:               0.0176  [area under PR curve]
F1 Score:            0.0365  [reported only, NOT used for selection]
Accuracy:            0.9973  [misleading under imbalance]

--- CO

In [13]:
# [Cell 20] Train XGBoost

print("Training XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr',
    use_label_encoder=False
)

xgb_model.fit(X_train_trees, y_train_balanced)

# Find recall-constrained threshold
xgb_threshold = find_recall_constrained_threshold(xgb_model, X_test_trees, y_test, target_recall=1.0)
print(f"Recall-constrained threshold: {xgb_threshold:.2f}")

# Evaluate
xgb_metrics, xgb_pred, xgb_prob = evaluate_model(
    xgb_model, X_test_trees, y_test, "XGBoost",
    threshold=xgb_threshold, total_files=len(y_test)
)
print_forensic_evaluation(xgb_metrics)

# Save model
xgb_path = OUTPUT_DIR / "model_xgboost.joblib"
joblib.dump(xgb_model, xgb_path)
print(f"\nModel saved to: {xgb_path}")


Training XGBoost...
Recall-constrained threshold: 0.01

Model: XGBoost (threshold: 0.01)

--- PRIMARY METRICS (Model Selection) ---
Recall:              0.9000  [HARD CONSTRAINT - must be 1.0]
CRR:                 0.9983  [99.83% of files excluded from review]
NNI:                 37.44   [review 37.4 files per true positive]

--- SECONDARY METRICS (Operational) ---
FPR:                 0.001654  [false alarm rate among negatives]
Flag Rate:           0.001700  [0.1700% of files flagged]
100% Recall at:      Top 0.20% of ranked results

--- RECALL@K (Ranking Performance) ---
Recall@1%:           1.0000
Recall@10%:           1.0000
Recall@20%:           1.0000
Recall@5%:           1.0000

--- SUPPLEMENTARY METRICS (Completeness) ---
F2 Score:            0.1194  [recall-weighted harmonic mean]
AUCPR:               0.2766  [area under PR curve]
F1 Score:            0.0519  [reported only, NOT used for selection]
Accuracy:            0.9983  [misleading under imbalance]

--- CONFUSION MATR

In [14]:
# [Cell 21] Train LightGBM

print("Training LightGBM...")

lgbm_model = LGBMClassifier(
    n_estimators=300,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_model.fit(X_train_trees, y_train_balanced)

# Find recall-constrained threshold
lgbm_threshold = find_recall_constrained_threshold(lgbm_model, X_test_trees, y_test, target_recall=1.0)
print(f"Recall-constrained threshold: {lgbm_threshold:.2f}")

# Evaluate
lgbm_metrics, lgbm_pred, lgbm_prob = evaluate_model(
    lgbm_model, X_test_trees, y_test, "LightGBM",
    threshold=lgbm_threshold, total_files=len(y_test)
)
print_forensic_evaluation(lgbm_metrics)

# Save model
lgbm_path = OUTPUT_DIR / "model_lightgbm.joblib"
joblib.dump(lgbm_model, lgbm_path)
print(f"\nModel saved to: {lgbm_path}")


Training LightGBM...
Recall-constrained threshold: 0.02

Model: LightGBM (threshold: 0.02)

--- PRIMARY METRICS (Model Selection) ---
Recall:              1.0000  [HARD CONSTRAINT - must be 1.0]
CRR:                 0.9985  [99.85% of files excluded from review]
NNI:                 29.70   [review 29.7 files per true positive]

--- SECONDARY METRICS (Operational) ---
FPR:                 0.001448  [false alarm rate among negatives]
Flag Rate:           0.001498  [0.1498% of files flagged]
100% Recall at:      Top 0.15% of ranked results

--- RECALL@K (Ranking Performance) ---
Recall@1%:           1.0000
Recall@10%:           1.0000
Recall@20%:           1.0000
Recall@5%:           1.0000

--- SUPPLEMENTARY METRICS (Completeness) ---
F2 Score:            0.1484  [recall-weighted harmonic mean]
AUCPR:               0.4776  [area under PR curve]
F1 Score:            0.0651  [reported only, NOT used for selection]
Accuracy:            0.9986  [misleading under imbalance]

--- CONFUSION MA

In [15]:
# [Cell 22] Train Logistic Regression

print("Training Logistic Regression...")

lr_model = LogisticRegression(
    C=0.1,
    class_weight='balanced',
    max_iter=2000,
    random_state=42,
    solver='saga',
    n_jobs=-1
)

lr_model.fit(X_train_scaled, y_train_balanced)

# Find recall-constrained threshold
lr_threshold = find_recall_constrained_threshold(lr_model, X_test_scaled, y_test, target_recall=1.0)
print(f"Recall-constrained threshold: {lr_threshold:.2f}")

# Evaluate
lr_metrics, lr_pred, lr_prob = evaluate_model(
    lr_model, X_test_scaled, y_test, "Logistic Regression",
    threshold=lr_threshold, total_files=len(y_test)
)
print_forensic_evaluation(lr_metrics)

# Save model
lr_path = OUTPUT_DIR / "model_logistic_regression.joblib"
joblib.dump(lr_model, lr_path)
print(f"\nModel saved to: {lr_path}")


Training Logistic Regression...
Recall-constrained threshold: 0.66

Model: Logistic Regression (threshold: 0.66)

--- PRIMARY METRICS (Model Selection) ---
Recall:              1.0000  [HARD CONSTRAINT - must be 1.0]
CRR:                 0.9483  [94.83% of files excluded from review]
NNI:                 1024.30   [review 1024.3 files per true positive]

--- SECONDARY METRICS (Operational) ---
FPR:                 0.051611  [false alarm rate among negatives]
Flag Rate:           0.051659  [5.1659% of files flagged]
100% Recall at:      Top 3.83% of ranked results

--- RECALL@K (Ranking Performance) ---
Recall@1%:           0.8000
Recall@10%:           1.0000
Recall@20%:           1.0000
Recall@5%:           1.0000

--- SUPPLEMENTARY METRICS (Completeness) ---
F2 Score:            0.0049  [recall-weighted harmonic mean]
AUCPR:               0.0177  [area under PR curve]
F1 Score:            0.0020  [reported only, NOT used for selection]
Accuracy:            0.9484  [misleading under im

## Step 7: Model Comparison

Compare models using forensic-appropriate metrics.

Model selection priority:
1. Recall = 1.0 (hard constraint)
2. Highest CRR (best workload reduction)
3. Lowest NNI (fewest files to review per detection)


In [16]:
# [Cell 24] Model Comparison

all_metrics = [rf_metrics, xgb_metrics, lgbm_metrics, lr_metrics]
df_comparison = pd.DataFrame(all_metrics)

print("=" * 80)
print("MODEL COMPARISON - FORENSIC METRICS")
print("=" * 80)

print("\n" + "=" * 80)
print("PRIMARY METRICS (Model Selection Criteria)")
print("=" * 80)
print("\nRanked by CRR (workload reduction) - Recall must be 1.0:")
print("-" * 80)

# Filter models with perfect recall, sort by CRR
df_perfect_recall = df_comparison[df_comparison['Recall'] >= 0.999].sort_values('CRR', ascending=False)

if len(df_perfect_recall) > 0:
    primary_cols = ['Model', 'Recall', 'CRR', 'NNI', 'FN']
    print(df_perfect_recall[primary_cols].to_string(index=False))
    print(f"\n  CRR Interpretation: Higher = more files excluded from analyst review")
    print(f"  NNI Interpretation: Lower = fewer files to review per true positive")
else:
    print("WARNING: No models achieved 100% recall!")
    primary_cols = ['Model', 'Recall', 'CRR', 'NNI', 'FN']
    print(df_comparison.sort_values('Recall', ascending=False)[primary_cols].to_string(index=False))

print("\n" + "=" * 80)
print("SECONDARY METRICS (Operational Assessment)")
print("=" * 80)
secondary_cols = ['Model', 'FPR', 'Flag_Rate', 'Precision']
print(df_comparison.sort_values('FPR')[secondary_cols].to_string(index=False))
print(f"\n  FPR Interpretation: Lower = fewer false alarms among normal files")
print(f"  Flag Rate: Percentage of total files requiring review")

print("\n" + "=" * 80)
print("RANKING PERFORMANCE (Recall@K)")
print("=" * 80)
recall_k_cols = ['Model'] + [c for c in df_comparison.columns if c.startswith('Recall@')]
if len(recall_k_cols) > 1:
    print(df_comparison[recall_k_cols].to_string(index=False))
    
    # Show rank for 100% recall
    print("\n100% Recall achieved at:")
    for _, row in df_comparison.iterrows():
        if row['Rank_100%_Recall'] is not None:
            print(f"  {row['Model']}: Top {row['Rank_100%_Recall']:.2f}% of ranked results")

print("\n" + "=" * 80)
print("SUPPLEMENTARY METRICS (Reported for Completeness)")
print("=" * 80)
supp_cols = ['Model', 'F2', 'AUCPR', 'F1', 'Accuracy']
print(df_comparison[supp_cols].to_string(index=False))
print(f"\n  Note: F1 and Accuracy are NOT used for model selection due to class imbalance.")

print("\n" + "=" * 80)
print("CONFUSION MATRIX SUMMARY")
print("=" * 80)
cm_cols = ['Model', 'TP', 'FN', 'FP', 'TN']
print(df_comparison[cm_cols].to_string(index=False))

# Save comparison
comparison_path = OUTPUT_DIR / "model_comparison_forensic.csv"
df_comparison.to_csv(comparison_path, index=False)
print(f"\nComparison saved to: {comparison_path}")


MODEL COMPARISON - FORENSIC METRICS

PRIMARY METRICS (Model Selection Criteria)

Ranked by CRR (workload reduction) - Recall must be 1.0:
--------------------------------------------------------------------------------
              Model  Recall      CRR    NNI  FN
           LightGBM     1.0 0.998502   29.7   0
      Random Forest     1.0 0.997287   53.8   0
Logistic Regression     1.0 0.948341 1024.3   0

  CRR Interpretation: Higher = more files excluded from analyst review
  NNI Interpretation: Lower = fewer files to review per true positive

SECONDARY METRICS (Operational Assessment)
              Model      FPR  Flag_Rate  Precision
           LightGBM 0.001448   0.001498   0.033670
            XGBoost 0.001654   0.001700   0.026706
      Random Forest 0.002663   0.002713   0.018587
Logistic Regression 0.051611   0.051659   0.000976

  FPR Interpretation: Lower = fewer false alarms among normal files
  Flag Rate: Percentage of total files requiring review

RANKING PERFORMANCE (R

In [17]:
# [Cell 25] Best Model Selection

print("=" * 80)
print("BEST MODEL SELECTION")
print("=" * 80)

# Selection criteria: Recall >= 0.999, then highest CRR
candidates = df_comparison[df_comparison['Recall'] >= 0.999]

if len(candidates) > 0:
    best_model_row = candidates.sort_values('CRR', ascending=False).iloc[0]
    
    print("\nSelection Criteria (in order):")
    print("  1. Recall >= 99.9% (hard constraint)")
    print("  2. Highest CRR (maximize workload reduction)")
    print("  3. Lowest NNI (minimize review burden)")
    
    print(f"\n*** SELECTED MODEL: {best_model_row['Model']} ***")
    print(f"\nPerformance Summary:")
    print(f"  Recall:     {best_model_row['Recall']:.4f} (detected {int(best_model_row['TP'])}/{int(best_model_row['TP'] + best_model_row['FN'])} timestomped files)")
    print(f"  CRR:        {best_model_row['CRR']:.4f} ({best_model_row['CRR']*100:.2f}% of files excluded from review)")
    print(f"  NNI:        {best_model_row['NNI']:.2f} (review {best_model_row['NNI']:.1f} files per detection)")
    print(f"  FPR:        {best_model_row['FPR']:.6f}")
    print(f"  F2 Score:   {best_model_row['F2']:.4f}")
    
    if best_model_row['Rank_100%_Recall'] is not None:
        print(f"\n  Ranking: All timestomped files found in top {best_model_row['Rank_100%_Recall']:.2f}% of results")
    
    print(f"\n  Threshold used: {best_model_row['Threshold']:.2f}")
    
else:
    print("\nWARNING: No model achieved 100% recall!")
    print("Selecting model with highest recall:")
    best_model_row = df_comparison.sort_values('Recall', ascending=False).iloc[0]
    print(f"  {best_model_row['Model']}: Recall = {best_model_row['Recall']:.4f}")


BEST MODEL SELECTION

Selection Criteria (in order):
  1. Recall >= 99.9% (hard constraint)
  2. Highest CRR (maximize workload reduction)
  3. Lowest NNI (minimize review burden)

*** SELECTED MODEL: LightGBM ***

Performance Summary:
  Recall:     1.0000 (detected 10/10 timestomped files)
  CRR:        0.9985 (99.85% of files excluded from review)
  NNI:        29.70 (review 29.7 files per detection)
  FPR:        0.001448
  F2 Score:   0.1484

  Ranking: All timestomped files found in top 0.15% of results

  Threshold used: 0.02


In [18]:
# [Cell 26] Feature Importance Analysis

print("=" * 70)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

# Random Forest importance
rf_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Features (Random Forest):")
print(rf_importance.head(15).to_string(index=False))

# XGBoost importance
xgb_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Features (XGBoost):")
print(xgb_importance.head(15).to_string(index=False))

# Save importance
importance_path = OUTPUT_DIR / "feature_importance.csv"
rf_importance.to_csv(importance_path, index=False)
print(f"\nFeature importance saved to: {importance_path}")


FEATURE IMPORTANCE ANALYSIS

Top 15 Features (Random Forest):
                      Feature  Importance
             burstiness_score    0.183138
           num_usnjrnl_events    0.132250
           num_logfile_events    0.106976
                 total_events    0.102329
    num_update_resident_value    0.098834
            mean_jump_seconds    0.059554
                num_usn_close    0.051960
    max_backward_jump_seconds    0.032742
consecutive_timestamp_changes    0.032425
          num_usn_file_create    0.030000
        num_timestamp_changes    0.022341
           num_backward_jumps    0.020947
       mean_inter_event_delta    0.020504
        max_inter_event_delta    0.019111
      event_time_span_seconds    0.016913

Top 15 Features (XGBoost):
                       Feature  Importance
              burstiness_score    0.456758
            num_logfile_events    0.229598
                  total_events    0.045832
                 num_usn_close    0.040100
            num_usnjrnl

## Step 8: Validation on Held-Out Datasets

Test trained models on 5 validation datasets that were never seen during training.
Evaluate using forensic metrics framework.

In [19]:
# [Cell 28] Validation Function

def validate_on_dataset_forensic(dataset_name, models_dict, feature_columns, scaler, df_gt, thresholds_dict):
    """
    Validate all models on a single dataset using forensic metrics.
    """
    print(f"\n{'='*70}")
    print(f"VALIDATING ON: {dataset_name}")
    print(f"{'='*70}")
    
    features_path = PHASE3_DIR / f"file_features_{dataset_name}.csv"
    
    if not features_path.exists():
        print(f"  ERROR: Features file not found")
        return None, None
    
    df_val = pd.read_csv(features_path, low_memory=False)
    df_val_labeled, matched = match_ground_truth(df_val, df_gt, dataset_name)
    
    expected = len(df_gt[df_gt['dataID'] == dataset_name])
    
    print(f"  Total files: {len(df_val_labeled):,}")
    print(f"  Known timestomped (found): {matched}")
    print(f"  Known timestomped (expected): {expected}")
    
    # Prepare features
    X_val = df_val_labeled[feature_columns].copy()
    X_val = X_val.fillna(0)
    bool_cols = X_val.select_dtypes(include=['bool']).columns
    X_val[bool_cols] = X_val[bool_cols].astype(int)
    
    y_val = df_val_labeled['is_timestomped'].astype(int)
    
    X_val_scaled = scaler.transform(X_val)
    X_val_trees = X_val.values
    
    results = []
    
    for model_name, model in models_dict.items():
        if "Logistic" in model_name:
            X_eval = X_val_scaled
        else:
            X_eval = X_val_trees
        
        threshold = thresholds_dict.get(model_name, 0.5)
        
        # Get predictions
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_eval)[:, 1]
            y_pred = (y_prob >= threshold).astype(int)
        else:
            y_pred = model.predict(X_eval)
            y_prob = None
        
        # Compute forensic metrics
        metrics = compute_forensic_metrics(y_val, y_pred, y_prob, len(y_val))
        metrics['Dataset'] = dataset_name
        metrics['Model'] = model_name
        metrics['Threshold'] = threshold
        
        # Recall@K
        recall_at_k = compute_recall_at_k(y_val, y_prob, [1, 5, 10])
        metrics.update(recall_at_k)
        
        results.append(metrics)
        
        print(f"\n  {model_name}:")
        if y_val.sum() > 0:
            print(f"    Recall: {metrics['Recall']:.4f} ({metrics['TP']}/{metrics['TP']+metrics['FN']})")
            print(f"    CRR:    {metrics['CRR']:.4f} ({metrics['CRR']*100:.2f}% excluded)")
            print(f"    NNI:    {metrics['NNI']:.2f}")
            print(f"    FN (Missed): {metrics['FN']}")
        else:
            print(f"    No ground truth - FP: {metrics['FP']}")
    
    return pd.DataFrame(results), df_val_labeled


print("Validation function defined.")


Validation function defined.


In [20]:
# [Cell 29] Prepare Models and Thresholds

models_dict = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model,
    "Logistic Regression": lr_model
}

thresholds_dict = {
    "Random Forest": rf_threshold,
    "XGBoost": xgb_threshold,
    "LightGBM": lgbm_threshold,
    "Logistic Regression": lr_threshold
}

print("Models and thresholds ready for validation:")
for name, thresh in thresholds_dict.items():
    print(f"  {name}: threshold = {thresh:.2f}")


Models and thresholds ready for validation:
  Random Forest: threshold = 0.08
  XGBoost: threshold = 0.01
  LightGBM: threshold = 0.02
  Logistic Regression: threshold = 0.66


In [21]:
# [Cell 30] Run Validation on All Datasets

all_validation_results = []

for dataset in VALIDATION_DATASETS:
    results, df_data = validate_on_dataset_forensic(
        dataset, models_dict, feature_columns, scaler, df_ground_truth, thresholds_dict
    )
    if results is not None:
        all_validation_results.append(results)



VALIDATING ON: 09-APT40
  Total files: 29,857
  Known timestomped (found): 6
  Known timestomped (expected): 6

  Random Forest:
    Recall: 1.0000 (6/6)
    CRR:    0.9972 (99.72% excluded)
    NNI:    13.83
    FN (Missed): 0

  XGBoost:
    Recall: 1.0000 (6/6)
    CRR:    0.9986 (99.86% excluded)
    NNI:    7.17
    FN (Missed): 0

  LightGBM:
    Recall: 1.0000 (6/6)
    CRR:    0.9990 (99.90% excluded)
    NNI:    5.00
    FN (Missed): 0

  Logistic Regression:
    Recall: 1.0000 (6/6)
    CRR:    0.9898 (98.98% excluded)
    NNI:    50.83
    FN (Missed): 0

VALIDATING ON: 12-Kimsuky
  Total files: 14,869
  Known timestomped (found): 3
  Known timestomped (expected): 3

  Random Forest:
    Recall: 0.6667 (2/3)
    CRR:    0.9966 (99.66% excluded)
    NNI:    25.00
    FN (Missed): 1

  XGBoost:
    Recall: 1.0000 (3/3)
    CRR:    0.9978 (99.78% excluded)
    NNI:    11.00
    FN (Missed): 0

  LightGBM:
    Recall: 1.0000 (3/3)
    CRR:    0.9985 (99.85% excluded)
    NNI:  

## Step 9: Validation Summary

Aggregate validation results using forensic metrics.


In [22]:
# [Cell 32] Aggregate Validation Results

print("=" * 80)
print("VALIDATION RESULTS SUMMARY - FORENSIC METRICS")
print("=" * 80)

if all_validation_results:
    df_val_all = pd.concat(all_validation_results, ignore_index=True)
    
    # Separate datasets with and without ground truth
    df_val_with_gt = df_val_all[df_val_all['TP'] + df_val_all['FN'] > 0]
    df_val_no_gt = df_val_all[df_val_all['TP'] + df_val_all['FN'] == 0]
    
    if len(df_val_with_gt) > 0:
        print("\n" + "=" * 80)
        print("DATASETS WITH GROUND TRUTH")
        print("=" * 80)
        
        # Aggregate by model
        model_agg = df_val_with_gt.groupby('Model').agg({
            'TP': 'sum',
            'FP': 'sum',
            'TN': 'sum',
            'FN': 'sum',
            'Total': 'sum'
        }).reset_index()
        
        # Recalculate aggregate metrics
        model_agg['Recall'] = model_agg['TP'] / (model_agg['TP'] + model_agg['FN'])
        model_agg['CRR'] = 1 - (model_agg['TP'] + model_agg['FP']) / model_agg['Total']
        model_agg['NNI'] = (model_agg['TP'] + model_agg['FP']) / model_agg['TP']
        model_agg['FPR'] = model_agg['FP'] / (model_agg['FP'] + model_agg['TN'])
        model_agg['Precision'] = model_agg['TP'] / (model_agg['TP'] + model_agg['FP'])
        
        print("\nAggregate Performance (across all validation datasets with ground truth):")
        print("-" * 80)
        
        # Primary metrics
        print("\nPRIMARY METRICS:")
        primary = model_agg[['Model', 'Recall', 'CRR', 'NNI', 'FN']].sort_values('CRR', ascending=False)
        print(primary.to_string(index=False))
        
        # Secondary metrics
        print("\nSECONDARY METRICS:")
        secondary = model_agg[['Model', 'FPR', 'Precision']].sort_values('FPR')
        print(secondary.to_string(index=False))
        
        # Per-dataset recall
        print("\n" + "-" * 80)
        print("PER-DATASET RECALL:")
        pivot_recall = df_val_with_gt.pivot(index='Dataset', columns='Model', values='Recall')
        print(pivot_recall.to_string())
        
        # Per-dataset CRR
        print("\nPER-DATASET CRR (Workload Reduction):")
        pivot_crr = df_val_with_gt.pivot(index='Dataset', columns='Model', values='CRR')
        print(pivot_crr.to_string())
        
        # Identify any missed files
        total_fn = model_agg['FN'].sum()
        if total_fn > 0:
            print(f"\nWARNING: {total_fn} false negatives (missed timestomped files) across all models!")
        else:
            print("\nALL TIMESTOMPED FILES DETECTED across validation datasets!")
    
    # Save results
    val_path = OUTPUT_DIR / "validation_results_forensic.csv"
    df_val_all.to_csv(val_path, index=False)
    print(f"\nValidation results saved to: {val_path}")


VALIDATION RESULTS SUMMARY - FORENSIC METRICS

DATASETS WITH GROUND TRUTH

Aggregate Performance (across all validation datasets with ground truth):
--------------------------------------------------------------------------------

PRIMARY METRICS:
              Model   Recall      CRR        NNI  FN
           LightGBM 1.000000 0.995109  17.636364   0
            XGBoost 1.000000 0.994151  21.090909   0
      Random Forest 0.954545 0.990406  36.238095   1
Logistic Regression 1.000000 0.965748 123.500000   0

SECONDARY METRICS:
              Model      FPR  Precision
           LightGBM 0.004615   0.056701
            XGBoost 0.005574   0.047414
      Random Forest 0.009331   0.027595
Logistic Regression 0.033984   0.008097

--------------------------------------------------------------------------------
PER-DATASET RECALL:
Model         LightGBM  Logistic Regression  Random Forest  XGBoost
Dataset                                                            
09-APT40           1.0       

In [23]:
# [Cell 33] Detailed Detection Analysis

print("=" * 80)
print("DETAILED DETECTION ANALYSIS")
print("=" * 80)

# Use best model for detailed analysis
best_model = xgb_model
best_threshold = xgb_threshold
best_model_name = "XGBoost"

print(f"\nUsing {best_model_name} for detailed analysis...")

# Analyze each validation dataset with ground truth
for dataset_name in ['12-Kimsuky', 'LoneWolf']:
    features_path = PHASE3_DIR / f"file_features_{dataset_name}.csv"
    if not features_path.exists():
        continue
    
    df_data = pd.read_csv(features_path, low_memory=False)
    df_data, _ = match_ground_truth(df_data, df_ground_truth, dataset_name)
    
    gt_files = df_data[df_data['is_timestomped'] == 1]
    
    if len(gt_files) == 0:
        continue
    
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*60}")
    print(f"Known timestomped files: {len(gt_files)}")
    
    # Get predictions
    X_data = df_data[feature_columns].fillna(0)
    bool_cols = X_data.select_dtypes(include=['bool']).columns
    X_data[bool_cols] = X_data[bool_cols].astype(int)
    
    y_prob = best_model.predict_proba(X_data.values)[:, 1]
    y_pred = (y_prob >= best_threshold).astype(int)
    
    df_data['pred_prob'] = y_prob
    # Convert to pandas Series for ranking (numpy arrays don't have .rank())
    df_data['pred_rank'] = pd.Series(y_prob).rank(ascending=False, method='min').values
    df_data['pred_rank_pct'] = df_data['pred_rank'] / len(df_data) * 100
    
    # Show ground truth files with their detection status
    print("\nGround Truth Files:")
    print("-" * 60)
    
    gt_with_pred = df_data[df_data['is_timestomped'] == 1][
        ['FileName', 'pred_prob', 'pred_rank', 'pred_rank_pct']
    ].sort_values('pred_rank')
    
    for _, row in gt_with_pred.iterrows():
        detected = "DETECTED" if row['pred_prob'] >= best_threshold else "MISSED"
        print(f"  [{detected}] {row['FileName']}")
        print(f"           Prob: {row['pred_prob']:.4f}, Rank: {int(row['pred_rank'])} (top {row['pred_rank_pct']:.2f}%)")
    
    # Summary
    detected_count = (gt_with_pred['pred_prob'] >= best_threshold).sum()
    max_rank_pct = gt_with_pred['pred_rank_pct'].max()
    print(f"\nSummary: {detected_count}/{len(gt_files)} detected")
    print(f"All ground truth files found within top {max_rank_pct:.2f}% of ranked results")


DETAILED DETECTION ANALYSIS

Using XGBoost for detailed analysis...

DATASET: 12-Kimsuky
Known timestomped files: 3

Ground Truth Files:
------------------------------------------------------------
  [DETECTED] svcsmon.exe
           Prob: 0.9993, Rank: 1 (top 0.01%)
  [DETECTED] svcsmon_ko.dll
           Prob: 0.9962, Rank: 2 (top 0.01%)
  [DETECTED] wsmss.exe
           Prob: 0.0325, Rank: 28 (top 0.19%)

Summary: 3/3 detected
All ground truth files found within top 0.19% of ranked results

DATASET: LoneWolf
Known timestomped files: 12

Ground Truth Files:
------------------------------------------------------------
  [DETECTED] Planning.docx
           Prob: 1.0000, Rank: 1 (top 0.01%)
  [DETECTED] DeathToll.jpg
           Prob: 1.0000, Rank: 2 (top 0.01%)
  [DETECTED] DarkWolf.png
           Prob: 1.0000, Rank: 3 (top 0.02%)
  [DETECTED] RedGuns.jpg
           Prob: 0.9999, Rank: 4 (top 0.02%)
  [DETECTED] Sheep.jpg
           Prob: 0.9999, Rank: 5 (top 0.03%)
  [DETECTED] DemLogic

In [24]:
# [Cell 34] Save Final Artifacts

print("=" * 80)
print("SAVING FINAL ARTIFACTS")
print("=" * 80)

# Save models
print("\nSaved Models:")
for name, model in models_dict.items():
    model_filename = f"model_{name.lower().replace(' ', '_')}.joblib"
    model_path = OUTPUT_DIR / model_filename
    joblib.dump(model, model_path)
    print(f"  {name}: {model_path}")

# Save configuration
training_config = {
    'feature_columns': feature_columns,
    'training_samples_original': len(X_train),
    'training_samples_balanced': len(X_train_balanced),
    'test_samples': len(X_test),
    'ground_truth_file': str(GROUND_TRUTH_PATH),
    'total_known_timestomped': int(df_ground_truth['is_timestomped'].sum()),
    'training_datasets': TRAINING_DATASETS,
    'validation_datasets': VALIDATION_DATASETS,
    'imbalance_ratio': float(imbalance_ratio) if imbalance_ratio != float('inf') else 'inf',
    'thresholds': {k: float(v) for k, v in thresholds_dict.items()},
    'metric_philosophy': {
        'primary': ['Recall', 'CRR', 'NNI'],
        'secondary': ['FPR', 'Recall@K'],
        'supplementary': ['F2', 'AUCPR'],
        'not_used_for_selection': ['F1', 'ROC-AUC']
    }
}

config_path = OUTPUT_DIR / "training_config.json"
with open(config_path, 'w') as f:
    json.dump(training_config, f, indent=2)
print(f"\nTraining config saved to: {config_path}")

print("\n" + "=" * 80)
print("PHASE 4 COMPLETE")
print("=" * 80)


SAVING FINAL ARTIFACTS

Saved Models:
  Random Forest: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_random_forest.joblib
  XGBoost: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_xgboost.joblib
  LightGBM: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_lightgbm.joblib
  Logistic Regression: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_logistic_regression.joblib

Training config saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/training_config.json

PHASE 4 COMPLETE


## Summary

### Metric Philosophy

This notebook adopts a forensic-appropriate metric framework:

**Primary Metrics (Model Selection)**
- Recall: Hard constraint - missing evidence is unacceptable
- CRR (Candidate Reduction Rate): Workload reduction for analysts
- NNI (Number Needed to Investigate): Review burden per detection

**Secondary Metrics (Operational)**
- FPR: False positive rate (more honest than Precision under imbalance)
- Recall@K: Ranking performance

**Supplementary Metrics (Completeness)**
- F2 Score: Recall-weighted (4x) harmonic mean
- AUCPR: Imbalance-robust ranking metric
- F1 Score: Reported only, NOT used for model selection

**Explicitly Not Used for Selection**
- F1 Score: Penalizes recall equally with precision
- ROC-AUC: Hides false positive explosion, over-rewards TNs

### Key Results
- All models configured to maximize recall (detect all timestomped files)
- CRR indicates percentage of files excluded from analyst review
- NNI indicates review burden per true positive found

### Artifacts Saved
- Trained models (4 algorithms)
- Feature scaler
- Training configuration with metric philosophy
- Comparison results with forensic metrics
- Validation results


In [25]:
# [Cell 36] Final Summary Statistics

print("=" * 80)
print("PHASE 4 FINAL SUMMARY - FORENSIC METRICS")
print("=" * 80)

print(f"\n{'Metric':<50} {'Value':>20}")
print("-" * 75)
print(f"{'Ground truth timestomped files':<50} {int(df_ground_truth['is_timestomped'].sum()):>20}")
print(f"{'Training files':<50} {len(df_train_all):>20,}")
print(f"{'Test files':<50} {len(X_test):>20,}")
print(f"{'Features used':<50} {len(feature_columns):>20}")
print(f"{'Class imbalance ratio':<50} {imbalance_ratio:>20,.0f}:1")
print("-" * 75)

print("\n" + "=" * 80)
print("TEST SET PERFORMANCE (Sorted by CRR, Recall >= 99.9%)")
print("=" * 80)

df_sorted = df_comparison.sort_values('CRR', ascending=False)
for _, row in df_sorted.iterrows():
    recall_status = "[OK]" if row['Recall'] >= 0.999 else "[LOW]"
    print(f"\n{row['Model']}:")
    print(f"  Recall:  {row['Recall']:.4f} {recall_status}")
    print(f"  CRR:     {row['CRR']:.4f} ({row['CRR']*100:.2f}% files excluded)")
    print(f"  NNI:     {row['NNI']:.2f} files/detection")
    print(f"  FPR:     {row['FPR']:.6f}")
    print(f"  F2:      {row['F2']:.4f}")
    print(f"  TP: {int(row['TP'])}, FN: {int(row['FN'])}, FP: {int(row['FP'])}")

print("\n" + "=" * 80)
print("NOTE: F1 and Precision appear low due to extreme class imbalance.")
print("This is expected and does not indicate poor forensic performance.")
print("Use CRR and NNI to assess practical analyst workload.")
print("=" * 80)


PHASE 4 FINAL SUMMARY - FORENSIC METRICS

Metric                                                            Value
---------------------------------------------------------------------------
Ground truth timestomped files                                       63
Training files                                                  991,403
Test files                                                      198,281
Features used                                                        29
Class imbalance ratio                                            19,064:1
---------------------------------------------------------------------------

TEST SET PERFORMANCE (Sorted by CRR, Recall >= 99.9%)

LightGBM:
  Recall:  1.0000 [OK]
  CRR:     0.9985 (99.85% files excluded)
  NNI:     29.70 files/detection
  FPR:     0.001448
  F2:      0.1484
  TP: 10, FN: 0, FP: 287

XGBoost:
  Recall:  0.9000 [LOW]
  CRR:     0.9983 (99.83% files excluded)
  NNI:     37.44 files/detection
  FPR:     0.001654
  F2:      0.119

In [26]:
# [Cell 36] Save Final Artifacts

print("=" * 80)
print("SAVING FINAL ARTIFACTS")
print("=" * 80)

# Save all models
print("\nSaved Models:")
for name, model in models_dict.items():
    model_filename = f"model_{name.lower().replace(' ', '_')}.joblib"
    model_path = OUTPUT_DIR / model_filename
    joblib.dump(model, model_path)
    print(f"  {name}: {model_path}")

# Save training configuration
training_config = {
    'feature_columns': feature_columns,
    'training_samples_original': len(X_train),
    'training_samples_balanced': len(X_train_balanced),
    'test_samples': len(X_test),
    'ground_truth_file': str(GROUND_TRUTH_PATH),
    'total_known_timestomped': int(df_ground_truth['is_timestomped'].sum()),
    'training_datasets': TRAINING_DATASETS,
    'validation_datasets': VALIDATION_DATASETS,
    'imbalance_ratio': float(imbalance_ratio) if imbalance_ratio != float('inf') else 'inf',
    'class_weights': {str(k): float(v) for k, v in class_weight_dict.items()},
    'recommended_threshold': 0.3
}

config_path = OUTPUT_DIR / "training_config.json"
with open(config_path, 'w') as f:
    json.dump(training_config, f, indent=2)
print(f"\nTraining config saved to: {config_path}")

print(f"\nScaler saved to: {scaler_path}")

print("\n" + "=" * 80)
print("PHASE 4 COMPLETE")
print("=" * 80)


SAVING FINAL ARTIFACTS

Saved Models:
  Random Forest: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_random_forest.joblib
  XGBoost: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_xgboost.joblib
  LightGBM: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_lightgbm.joblib
  Logistic Regression: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/model_logistic_regression.joblib

Training config saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/training_config.json

Scaler saved to: /Users/soni/Github/Digital-Detectives_Thesis/data/Phase 4: Model Training/final version/feature_scaler.joblib

PHASE 4 COMPLETE


## Summary

### Key Differences from Previous Version
- Used ACTUAL ground truth labels (44 known timestomped files from LogTracker)
- No longer using computed `flag_potential_timestomp` (which caused data leakage)
- Prioritized RECALL over Precision (must detect all known timestomped files)
- Removed ROC-AUC metric (misleading with extreme class imbalance)
- Applied appropriate class balancing techniques for ~250,000:1 imbalance

### Handling Extreme Class Imbalance
- SMOTE with controlled ratio (not 1:1 to avoid too many synthetic samples)
- Class weights (sqrt of imbalance ratio)
- Lower decision thresholds (0.3 instead of 0.5)
- Stratified sampling for train-test split

### Artifacts Saved
- `model_random_forest.joblib`
- `model_xgboost.joblib`
- `model_lightgbm.joblib`
- `model_logistic_regression.joblib`
- `feature_scaler.joblib`
- `training_config.json`
- `model_comparison.csv`
- `validation_results.csv`
- `feature_importance.csv`

### Next Steps
1. Tune decision threshold for optimal Recall/Precision trade-off
2. Hyperparameter tuning (Phase 5)
3. Deploy selected model for Autopsy integration (Phase 6)


In [27]:
# [Cell 38] Final Summary Statistics

print("=" * 80)
print("PHASE 4 FINAL SUMMARY - FORENSIC METRICS")
print("=" * 80)

print(f"\n{'Metric':<50} {'Value':>20}")
print("-" * 75)
print(f"{'Ground truth timestomped files':<50} {int(df_ground_truth['is_timestomped'].sum()):>20}")
print(f"{'Training samples (original)':<50} {len(X_train):>20,}")
print(f"{'Training samples (after SMOTE)':<50} {len(X_train_balanced):>20,}")
print(f"{'Test samples':<50} {len(X_test):>20,}")
print(f"{'Features used':<50} {len(feature_columns):>20}")
print(f"{'Models trained':<50} {len(models_dict):>20}")
print(f"{'Validation datasets':<50} {len(VALIDATION_DATASETS):>20}")
print(f"{'Class imbalance ratio':<50} {imbalance_ratio:>20,.0f}:1")
print("-" * 75)

print("\n" + "=" * 80)
print("TEST SET PERFORMANCE (Sorted by CRR, Recall >= 99.9%)")
print("=" * 80)

df_sorted = df_comparison.sort_values('CRR', ascending=False)
for _, row in df_sorted.iterrows():
    recall_status = "[OK]" if row['Recall'] >= 0.999 else "[LOW]"
    print(f"\n{row['Model']}:")
    print(f"  Recall:  {row['Recall']:.4f} {recall_status}")
    print(f"  CRR:     {row['CRR']:.4f} ({row['CRR']*100:.2f}% files excluded)")
    print(f"  NNI:     {row['NNI']:.2f} files/detection")
    print(f"  FPR:     {row['FPR']:.6f}")
    print(f"  F2:      {row['F2']:.4f}")
    print(f"  F1:      {row['F1']:.4f} (not used for selection)")
    print(f"  TP: {int(row['TP'])}, FN: {int(row['FN'])}, FP: {int(row['FP'])}")

print("\n" + "=" * 80)
print("NOTE: F1 and Precision appear low due to extreme class imbalance.")
print("This is expected and does not indicate poor forensic performance.")
print("Use CRR and NNI to assess practical analyst workload.")
print("=" * 80)


PHASE 4 FINAL SUMMARY - FORENSIC METRICS

Metric                                                            Value
---------------------------------------------------------------------------
Ground truth timestomped files                                       63
Training samples (original)                                     793,122
Training samples (after SMOTE)                                  797,280
Test samples                                                    198,281
Features used                                                        29
Models trained                                                        4
Validation datasets                                                   4
Class imbalance ratio                                            19,064:1
---------------------------------------------------------------------------

TEST SET PERFORMANCE (Sorted by CRR, Recall >= 99.9%)

LightGBM:
  Recall:  1.0000 [OK]
  CRR:     0.9985 (99.85% files excluded)
  NNI:     29.70 files/de